In [2]:
from qdrant_client import QdrantClient, models
import requests

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-05 07:15:27.558395684 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [3]:
docs_url='https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json'
docs_response=requests.get(docs_url)
documents_raw=docs_response.json()

In [5]:
client=QdrantClient("http://127.0.0.1:6333/")

In [6]:
client.get_collections()

CollectionsResponse(collections=[CollectionDescription(name='zoomcamp-rag')])

#### Collections

In [7]:
client.create_collection(
    collection_name='zoomcamp-sparse',
    sparse_vectors_config={
        "bm25":models.SparseVectorParams(
            modifier=models.Modifier.IDF 
        )
    }
)

True

In [10]:
import uuid

client.upsert(
    collection_name='zoomcamp-sparse',
    points=[
        models.PointStruct(
            id=uuid.uuid4().hex,
            vector={
                "bm25":models.Document(
                    text=doc["text"],
                    model='Qdrant/bm25'
                )
            },
            payload={
                "text": doc["text"],
                "section": doc["section"],
                "course": course["course"]
            }
        )
        for course in documents_raw
        for doc in course["documents"]
    ]
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [11]:
def search(query: str, limit: int = 1) -> list[models.ScoredPoint]:
    results = client.query_points(
        collection_name="zoomcamp-sparse",
        query=models.Document(
            text=query,
            model="Qdrant/bm25",
        ),
        using="bm25",
        limit=limit,
        with_payload=True,
    )

    return results.points

In [30]:
results=search("mkdir")

In [33]:
results[0].score

8.058514

In [35]:
import random
import json
random.seed(202506)

course=random.choice(documents_raw)
course_piece=random.choice(course['documents'])
print(json.dumps(course_piece, indent=2))

{
  "text": "Even though the upload works using aws cli and boto3 in Jupyter notebook.\nSolution set the AWS_PROFILE environment variable (the default profile is called default)",
  "section": "Module 4: Deployment",
  "question": "Uploading to s3 fails with An error occurred (InvalidAccessKeyId) when calling the PutObject operation: The AWS Access Key Id you provided does not exist in our records.\""
}


In [41]:
print(search(course_piece['question'])[0].payload['text'])

You may get an error while creating a bucket with localstack and the boto3 client:
botocore.exceptions.ClientError: An error occurred (IllegalLocationConstraintException) when calling the CreateBucket operation: The unspecified location constraint is incompatible for the region specific endpoint this request was sent to.
To fix this, instead of creating a bucket via
s3_client.create_bucket(Bucket='nyc-duration')
Create it with
s3_client.create_bucket(Bucket='nyc-duration', CreateBucketConfiguration={
'LocationConstraint': AWS_DEFAULT_REGION})
yam
Added by M


#### Collection using both Dense and Sparse Vectors

In [44]:
client.create_collection(
    collection_name='zoomcamp-hybrid',
    vectors_config={
        'jina_small':models.VectorParams(
            size=512,
            distance=models.Distance.COSINE
        ),
    },
    sparse_vectors_config={
        'bm25':models.SparseVectorParams(
            modifier=models.Modifier.IDF
        )
    }
)

True

In [47]:
dense="jinaai/jina-embeddings-v2-small-en"

In [51]:
client.upsert(
    collection_name='zoomcamp-hybrid',
    points=[
        models.PointStruct(
            id=uuid.uuid4().hex,
            vector={
                "jina_small":models.Document(
                    text=doc['text'],
                    model=dense
                ),
                "bm25": models.Document(
                    text=doc['text'],
                    model='Qdrant/bm25'
                )
            },
            payload={
                "text":doc['text'],
                "section":doc['section'],
                "course":course['course']
            }
        )
        for course in documents_raw
        for doc in course['documents']
    ]
)

2026-05-05 08:12:26.339 | ERROR    | fastembed.common.model_management:download_model:444 - Could not download model from HuggingFace: (Amz CF ID: 5NI0cM7kWXTymAjM80oJ5KvE5ayBGFu6VjlhqMoFZHcvPg2vwX5TPg==)

429 Too Many Requests for url: https://huggingface.co/api/models/xenova/jina-embeddings-v2-small-en. Falling back to other sources.
2026-05-05 08:12:26.339 | ERROR    | fastembed.common.model_management:download_model:467 - Could not download model from either source, sleeping for 3.0 seconds, 2 retries left.
2026-05-05 08:12:29.344 | ERROR    | fastembed.common.model_management:download_model:444 - Could not download model from HuggingFace: (Amz CF ID: CxRYt0X9keVajt-wBL-iaThk7fmGXIV7bFKRnyyA8rO7z0f4nE1nnQ==)

429 Too Many Requests for url: https://huggingface.co/api/models/xenova/jina-embeddings-v2-small-en. Falling back to other sources.
2026-05-05 08:12:29.345 | ERROR    | fastembed.common.model_management:download_model:467 - Could not download model from either source, sleeping

KeyboardInterrupt: 